In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch import optim
import k3d
import sys
import os

import trimesh
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from models.model_architecture import GeneralNet
from util.visualization.utils_mesh import get_mesh

torch.manual_seed(0)


device = 'cuda' #if torch.cuda.is_available() else 'cpu'
torch.set_default_device(device)

In [2]:
mesh = trimesh.load("bun_zipper.ply")
# mesh = trimesh.load("rocker-arm.off")
pts = torch.tensor(mesh.vertices, dtype=torch.float64)
center = pts.mean(dim=0)
pts -= center
scale = pts.norm(dim=1).max()
pts /= scale
model = GeneralNet(ks=[3, 32, 32, 32, 1], act=torch.sin)
model.load_state_dict(torch.load('Trained Models/3,32,32,32,1,sin,bunny,adam', map_location=torch.device('cpu')))
model.double()

GeneralNet(
  (fcs): ModuleList(
    (0): Linear(in_features=3, out_features=32, bias=True)
    (1-2): 2 x Linear(in_features=32, out_features=32, bias=True)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [3]:
verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=pts.min(dim=0).values-0.01,
    bbox_max=pts.max(dim=0).values+0.15,
    chunks=2
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig.display()
model.double()

/cluster/home/jaking/.local/lib/python3.11/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "int32" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

GeneralNet(
  (fcs): ModuleList(
    (0): Linear(in_features=3, out_features=32, bias=True)
    (1-2): 2 x Linear(in_features=32, out_features=32, bias=True)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [4]:
fig_output_file = "../docs/k3d_plot_bunny_adam.html"
with open(fig_output_file, 'w') as f:
    f.write(fig.get_snapshot())
print(f"Plot saved to {fig_output_file}")

Plot saved to ../docs/k3d_plot_bunny_adam.html
